# Day 096 | Streamlit Dashboard for Geotech ML (Concept)

## 100 Days of ML for Geotechnical Engineering
**Phase:** Geotechnical AI Projects

**Author:** Ripon Chandra Malo

---

## Project Overview

**Objective:** Design and write a complete Streamlit web application that lets geotechnical engineers upload CSV data, select an ML model, train it, make predictions, and visualize results -- all through a browser interface.

**What This Notebook Covers:**
- The complete Streamlit app code (written to a `.py` file)
- How to structure a geotechnical ML dashboard
- Key Streamlit components: file upload, sidebar, selectbox, charts
- How to run the app locally

**Note:** Streamlit apps run as standalone Python scripts, not inside Jupyter. This notebook creates the app file and demonstrates the underlying logic.

---

## Part 1: Imports and Demo Data

In [ ]:
# ─────────────────────────────────────────────────────
# IMPORTS (for in-notebook demonstration)
# ─────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries loaded.")
print("\nNote: Streamlit must be installed separately:")
print("  pip install streamlit")
print("  streamlit run geotech_dashboard.py")

---

## Part 2: Generate Sample CSV Data

We create the sample CSV that users can upload to the dashboard.

In [ ]:
# ─────────────────────────────────────────────────────
# GENERATE SAMPLE GEOTECHNICAL DATA
# ─────────────────────────────────────────────────────

n = 200
depth = np.random.uniform(1, 30, n)
n60 = np.random.randint(2, 55, n).astype(float)
fines = np.random.uniform(0, 80, n)
water_content = np.random.uniform(10, 60, n)
unit_weight = np.random.uniform(14, 22, n)

# Target: undrained shear strength (synthetic relationship)
su = 5 * n60 + 0.3 * depth * unit_weight - 0.5 * water_content + np.random.normal(0, 10, n)
su = np.clip(su, 10, 500)

df_sample = pd.DataFrame({
    'Depth_m': np.round(depth, 1),
    'N60': n60,
    'Fines_pct': np.round(fines, 1),
    'Water_Content_pct': np.round(water_content, 1),
    'Unit_Weight_kN_m3': np.round(unit_weight, 2),
    'Su_kPa': np.round(su, 1),
})

print(f"Sample data: {len(df_sample)} rows")
print(f"\nThis CSV can be uploaded to the Streamlit dashboard.")
df_sample.head(10)

---

## Part 3: Dashboard Logic (Notebook Demonstration)

Before writing the Streamlit code, we demonstrate the underlying ML pipeline.

In [ ]:
# ─────────────────────────────────────────────────────
# ML PIPELINE (the logic behind the dashboard)
# ─────────────────────────────────────────────────────

# Step 1: Select features and target
feature_cols = ['Depth_m', 'N60', 'Fines_pct', 'Water_Content_pct', 'Unit_Weight_kN_m3']
target_col = 'Su_kPa'

X = df_sample[feature_cols].values
y = df_sample[target_col].values

# Step 2: Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 3: Scale
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Step 4: Train multiple models
models_demo = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
}

for name, model in models_demo.items():
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    print(f"{name:25s}: R2={r2:.4f}, RMSE={rmse:.2f} kPa")

In [ ]:
# ─────────────────────────────────────────────────────
# DASHBOARD VISUALIZATIONS (what the app will show)
# ─────────────────────────────────────────────────────

best_model = models_demo['Gradient Boosting']
y_pred = best_model.predict(X_test_s)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Parity plot
axes[0].scatter(y_test, y_pred, c='#2196F3', s=30, alpha=0.6)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'k--', alpha=0.5)
axes[0].set_xlabel('Actual Su (kPa)')
axes[0].set_ylabel('Predicted Su (kPa)')
axes[0].set_title('Predicted vs Actual', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Residuals
residuals = y_test - y_pred
axes[1].hist(residuals, bins=20, color='#FF9800', edgecolor='white')
axes[1].set_xlabel('Residual (kPa)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution', fontweight='bold')
axes[1].axvline(0, color='red', linestyle='--')

# Feature importance
importances = best_model.feature_importances_
fi_sorted = sorted(zip(feature_cols, importances), key=lambda x: x[1])
axes[2].barh([f[0] for f in fi_sorted], [f[1] for f in fi_sorted], 
             color='#4CAF50', edgecolor='white')
axes[2].set_xlabel('Importance')
axes[2].set_title('Feature Importance', fontweight='bold')

fig.suptitle('Dashboard Output Preview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Part 4: Complete Streamlit App Code

The cell below contains the full Streamlit application. To run it:

1. Save the code to a file called `geotech_dashboard.py`
2. Install Streamlit: `pip install streamlit`
3. Run: `streamlit run geotech_dashboard.py`
4. Open your browser to `http://localhost:8501`

In [ ]:
# ─────────────────────────────────────────────────────
# COMPLETE STREAMLIT APP CODE
# Save this to: geotech_dashboard.py
# Run with:     streamlit run geotech_dashboard.py
# ─────────────────────────────────────────────────────

streamlit_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler

# ── Page Configuration ──
st.set_page_config(
    page_title="Geotech ML Dashboard",
    page_icon="🏗️",
    layout="wide"
)

st.title("Geotechnical ML Dashboard")
st.markdown("Upload geotechnical data, train ML models, and make predictions.")

# ── Sidebar ──
st.sidebar.header("Configuration")

# ── Data Upload ──
st.sidebar.subheader("1. Upload Data")
uploaded_file = st.sidebar.file_uploader("Upload CSV", type=["csv"])

if uploaded_file is not None:
    df = pd.read_csv(uploaded_file)
else:
    # Generate demo data
    st.sidebar.info("Using demo data. Upload your CSV to use your own.")
    np.random.seed(42)
    n = 200
    depth = np.random.uniform(1, 30, n)
    n60 = np.random.randint(2, 55, n).astype(float)
    fines = np.random.uniform(0, 80, n)
    wc = np.random.uniform(10, 60, n)
    uw = np.random.uniform(14, 22, n)
    su = 5 * n60 + 0.3 * depth * uw - 0.5 * wc + np.random.normal(0, 10, n)
    su = np.clip(su, 10, 500)
    df = pd.DataFrame({
        "Depth_m": np.round(depth, 1),
        "N60": n60, "Fines_pct": np.round(fines, 1),
        "Water_Content_pct": np.round(wc, 1),
        "Unit_Weight_kN_m3": np.round(uw, 2),
        "Su_kPa": np.round(su, 1),
    })

# ── Data Preview ──
st.header("Data Preview")
col1, col2 = st.columns(2)
with col1:
    st.write(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
    st.dataframe(df.head(10), use_container_width=True)
with col2:
    st.write("Statistics")
    st.dataframe(df.describe().round(2), use_container_width=True)

# ── Feature/Target Selection ──
st.sidebar.subheader("2. Select Features & Target")
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
target_col = st.sidebar.selectbox("Target variable", numeric_cols,
                                   index=len(numeric_cols)-1)
feature_cols = st.sidebar.multiselect("Feature variables",
    [c for c in numeric_cols if c != target_col],
    default=[c for c in numeric_cols if c != target_col]
)

if len(feature_cols) == 0:
    st.warning("Select at least one feature.")
    st.stop()

# ── Model Selection ──
st.sidebar.subheader("3. Select Model")
model_name = st.sidebar.selectbox("ML Model", [
    "Linear Regression", "Random Forest", "Gradient Boosting"
])
test_size = st.sidebar.slider("Test size (%)", 10, 40, 20) / 100

# ── Train Model ──
if st.sidebar.button("Train Model"):
    X = df[feature_cols].values
    y = df[target_col].values

    # Remove NaN rows
    valid = ~(np.isnan(X).any(axis=1) | np.isnan(y))
    X, y = X[valid], y[valid]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42
    )

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    models = {
        "Linear Regression": LinearRegression(),
        "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
        "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
    }
    model = models[model_name]
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)

    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)

    # ── Results ──
    st.header("Model Results")

    m1, m2, m3 = st.columns(3)
    m1.metric("R2 Score", f"{r2:.4f}")
    m2.metric("RMSE", f"{rmse:.2f}")
    m3.metric("MAE", f"{mae:.2f}")

    # Charts
    chart1, chart2 = st.columns(2)

    with chart1:
        fig1, ax1 = plt.subplots(figsize=(6, 5))
        ax1.scatter(y_test, y_pred, alpha=0.6, s=20)
        lims = [min(y_test.min(), y_pred.min()),
                max(y_test.max(), y_pred.max())]
        ax1.plot(lims, lims, "k--", alpha=0.5)
        ax1.set_xlabel(f"Actual {target_col}")
        ax1.set_ylabel(f"Predicted {target_col}")
        ax1.set_title("Predicted vs Actual")
        ax1.grid(True, alpha=0.3)
        st.pyplot(fig1)

    with chart2:
        fig2, ax2 = plt.subplots(figsize=(6, 5))
        residuals = y_test - y_pred
        ax2.hist(residuals, bins=20, edgecolor="white")
        ax2.axvline(0, color="red", linestyle="--")
        ax2.set_xlabel("Residual")
        ax2.set_ylabel("Count")
        ax2.set_title("Residual Distribution")
        st.pyplot(fig2)

    # Feature importance (if available)
    if hasattr(model, "feature_importances_"):
        st.subheader("Feature Importance")
        fi = pd.DataFrame({
            "Feature": feature_cols,
            "Importance": model.feature_importances_
        }).sort_values("Importance", ascending=True)
        fig3, ax3 = plt.subplots(figsize=(8, 4))
        ax3.barh(fi["Feature"], fi["Importance"])
        ax3.set_xlabel("Importance")
        ax3.set_title("Feature Importance")
        st.pyplot(fig3)

    # ── Single Prediction ──
    st.header("Make a Prediction")
    pred_cols = st.columns(len(feature_cols))
    pred_values = []
    for i, col in enumerate(feature_cols):
        val = pred_cols[i].number_input(
            col, value=float(df[col].median()), key=f"pred_{col}"
        )
        pred_values.append(val)

    if st.button("Predict"):
        pred_input = scaler.transform([pred_values])
        pred_result = model.predict(pred_input)[0]
        st.success(f"Predicted {target_col}: {pred_result:.2f}")
'''

print("Streamlit app code ready.")
print(f"Total lines: {len(streamlit_code.strip().splitlines())}")
print("\nTo use this app:")
print("  1. Copy the code above to 'geotech_dashboard.py'")
print("  2. pip install streamlit")
print("  3. streamlit run geotech_dashboard.py")

---

## Part 5: App Architecture Walkthrough

In [ ]:
# ─────────────────────────────────────────────────────
# APP ARCHITECTURE
# ─────────────────────────────────────────────────────

architecture = """
Streamlit Dashboard Architecture
================================

SIDEBAR (Left Panel):
  +------------------------------+
  | 1. Upload CSV               |
  |    [File uploader]          |
  |                              |
  | 2. Select Features/Target    |
  |    [Dropdowns]              |
  |                              |
  | 3. Select Model              |
  |    [Linear/RF/GB]           |
  |    [Test size slider]       |
  |                              |
  | [Train Model Button]        |
  +------------------------------+

MAIN AREA (Right Panel):
  +------------------------------+
  | Data Preview                 |
  |   [DataFrame] [Statistics]  |
  +------------------------------+
  | Model Results                |
  |   R2=0.95  RMSE=8.5  MAE=6 |
  +------------------------------+
  | [Parity Plot] [Residuals]   |
  +------------------------------+
  | Feature Importance           |
  +------------------------------+
  | Make a Prediction            |
  |   [Input fields] [Predict]  |
  +------------------------------+
"""
print(architecture)

In [ ]:
# ─────────────────────────────────────────────────────
# KEY STREAMLIT COMPONENTS REFERENCE
# ─────────────────────────────────────────────────────

components = {
    'st.title()': 'Page title (large heading)',
    'st.header()': 'Section heading',
    'st.sidebar': 'Left panel for controls',
    'st.file_uploader()': 'CSV/file upload widget',
    'st.selectbox()': 'Dropdown selection',
    'st.multiselect()': 'Multi-choice selector',
    'st.slider()': 'Numeric slider',
    'st.number_input()': 'Numeric text input',
    'st.button()': 'Clickable button',
    'st.columns()': 'Side-by-side layout',
    'st.metric()': 'KPI metric display (value + delta)',
    'st.dataframe()': 'Interactive data table',
    'st.pyplot()': 'Matplotlib figure display',
    'st.success()': 'Green success message',
    'st.warning()': 'Yellow warning message',
    'st.info()': 'Blue info message',
}

print("Streamlit Components Used in the Dashboard:")
print("=" * 55)
for comp, desc in components.items():
    print(f"  {comp:25s}  {desc}")

---

## Part 6: Extending the Dashboard

Ideas for extending this basic dashboard into a production tool.

In [ ]:
# ─────────────────────────────────────────────────────
# EXTENSION IDEAS
# ─────────────────────────────────────────────────────

extensions = """
Dashboard Extension Ideas
=========================

1. DATA QUALITY TAB
   - Missing value heatmap
   - Outlier detection with IQR
   - Correlation matrix
   - Data profiling report

2. MULTI-MODEL COMPARISON
   - Train all models at once
   - Side-by-side metric comparison
   - Ensemble prediction option

3. CROSS-VALIDATION
   - K-Fold CV with confidence intervals
   - Learning curves
   - Hyperparameter tuning (GridSearchCV)

4. GEOTECHNICAL VISUALIZATIONS
   - N60 vs depth profile
   - Plasticity chart (Casagrande)
   - Soil profile column
   - Borehole location map

5. EXPORT
   - Download predictions as CSV
   - Generate PDF report
   - Save trained model (joblib)

6. DEPLOYMENT
   - Streamlit Cloud (free hosting)
   - Docker container
   - Add authentication
"""
print(extensions)

In [ ]:
# ─────────────────────────────────────────────────────
# DEPLOYMENT INSTRUCTIONS
# ─────────────────────────────────────────────────────

print("Deployment Options")
print("=" * 50)

print("""
OPTION 1: Local (Development)
  pip install streamlit
  streamlit run geotech_dashboard.py
  -> Opens at http://localhost:8501

OPTION 2: Streamlit Community Cloud (Free)
  1. Push code to GitHub
  2. Go to share.streamlit.io
  3. Connect your repo
  4. Deploy with one click

  Required files:
    geotech_dashboard.py
    requirements.txt:
      streamlit
      pandas
      numpy
      scikit-learn
      matplotlib

OPTION 3: Docker
  FROM python:3.9-slim
  COPY requirements.txt .
  RUN pip install -r requirements.txt
  COPY geotech_dashboard.py .
  CMD ["streamlit", "run", "geotech_dashboard.py"]
""")

---

## Key Takeaways

1. **Streamlit turns Python scripts into web apps** with minimal effort. No HTML/CSS/JavaScript required. A complete ML dashboard can be built in under 150 lines of Python.

2. **The sidebar/main layout** naturally separates controls (data upload, model selection) from results (metrics, charts, predictions).

3. **Interactive widgets** (file upload, dropdowns, sliders) let non-technical engineers use ML models without writing code.

4. **Streamlit Community Cloud provides free hosting** -- push to GitHub, connect, deploy. Your geotechnical team can access it from anywhere.

5. **Start simple, extend later.** The basic dashboard (upload, train, predict) is useful immediately. Add data quality checks, CV, and export features as needed.

---

## Next Steps

- Day 097: Full end-to-end ML project for pile bearing capacity
- Add multi-page navigation with `st.tabs()`
- Integrate the data pipeline from Day 094

---

*Day 096 of 100 -- From notebook to web app in under 150 lines!*